# 01 · 存档旧项目：论文那一版随时能找回

## 这一章在整个项目里的位置

```
00 总览 → [01 存档旧项目 ✅] → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 起：固定流程、agent …
```

> **这一章 10 月 7 日已经做完了。** 下面的命令只是查看，不改任何东西，重复运行安全。

## 背景：为什么动手之前先存档

旧项目（GitHub 上公开的 `pipi0616/OtoVCE`）是 OtoVCE 论文的代码。审稿期间，审稿人可能要求复现结果，**论文那一版必须能原样拿出来**。

动手前我先查了旧项目的状态，发现有三层：

| 层 | 在哪 | 情况 |
|---|---|---|
| 1 | GitHub 上 | 停在 7 月 20 日。这是读者看到的"论文代码" |
| 2 | 你电脑上的正本 | 比 GitHub 多 5 次 8 月 2 日的提交，**从没推上去** |
| 3 | 你电脑上没存的改动 | 8 月 5–21 日的工作，37 处改动、5.5 MB，**完全没存进记录** |

第 2、3 层只存在于你的硬盘上。第 3 层更危险：没存进记录的改动，一条命令就可能被抹掉。

**公司里的做法**：大改之前，先给当前版本打一个标签，把没合并的工作存进分支，然后再动手。什么都不删。

## 这一章做的 3 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 给论文那一版钉一个永久名字 | git 标签（tag） | 以后怎么改，都能一键回到那一版 |
| 2 | 8 月的工作存进"抽屉" | git 分支（branch） | 不丢，也不碰正本 |
| 3 | 旧项目和新项目分开放 | 两个独立的仓库 | 两份记录互不干扰 |

**花费：0。**

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OLD = ROOT.parent / "old"
os.chdir(OLD)
os.environ.update(NO_COLOR="1", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("旧项目目录:", OLD)

---

## 第 1 步 · 给论文那一版钉一个永久名字：标签

### 在干什么
给 GitHub 上那一版（7 月 20 日）起一个永久的名字：`paper-v1`。

### 背景
代码会一直改。半年后审稿人问"论文里的结果是哪一版代码跑的"，你要能一键回到那一版，而不是翻聊天记录、猜日期。

### 用了什么技术，为什么
**标签**（tag）：给某一次提交起一个永久的名字。提交的编号是一串字母数字（比如 `c84508d`），记不住；标签是人能记住的名字。

标签钉在 **GitHub 上那一版**，而不是你电脑上多出来的 5 次提交上：读者在 GitHub 上看到的才是"论文代码"，标签要和它一致。

当时运行的命令（已经做过，不用再运行）：

```bash
git tag -a paper-v1 c84508d -m "Code state of the OtoVCE paper as published on GitHub"
```

### 公司里什么情况怎么做
- **每次发布版本都打标签**：`v1.0`、`v1.1`……线上出了问题，"回到 v1.0"是一条命令
- **标签打了就不改**：要改，就打一个新的（比如 `paper-v2`），旧的留着

### 执行

In [ ]:
run("git tag -n1")

### 结果怎么看
- `paper-v1` 后面跟着说明 `Code state of the OtoVCE paper ...`：这就是论文那一版
- 另外三个 `v1.0`、`v1.0.1`、`v1.1`，是以前发布时打的

想回到论文那一版看代码：`git switch --detach paper-v1`；看完 `git switch main` 回来。（现在不用做。）

### 面试怎么说
> "Before refactoring I tagged the published state, so the paper's results stay reproducible no matter what changes later."

---

## 第 2 步 · 8 月的工作存进"抽屉"：分支

### 在干什么
把第 2、3 层（5 次没推的提交 + 两个月没存的改动）全部存进一个单独的分支：`archive/agent-experiments-2026-08`。

### 背景
这些工作（curator 文献整理 agent 约一万行、诊断 agent 的几个演示、评测结果）新项目暂时用不上，但不能丢：里面有做过的实验和踩过的坑。留在电脑上不存，迟早会被误删或覆盖。

### 用了什么技术，为什么
**分支**（branch）：正本的一个副本，在分支上怎么改，都不影响正本 `main`。

比喻：正本是实验记录本，分支是复印出来的一份，可以在复印件上随便写；要不要并回记录本，另外决定。这里只是把它当**抽屉**用：存进去，不并回去。

当时运行的命令（已经做过）：

```bash
git switch -c archive/agent-experiments-2026-08   # 开一个新分支，并切换过去
git add -A && git commit -m "archive: ..."         # 把所有没存的改动存进去
git switch main                                    # 回到正本
```

### 公司里什么情况怎么做
- **每一个改动都在分支上做**：做完提合并请求（04 号讲），审查通过才并回正本。没有人直接改正本
- **分支名说明用途**：`feat/...` 新功能、`fix/...` 修错、`archive/...` 存档，看名字就知道是干什么的

### 执行

In [ ]:
run("git branch")
run("git log --oneline -1 --stat archive/agent-experiments-2026-08 | tail -1")

### 结果怎么看
- 两个分支：`archive/agent-experiments-2026-08`（抽屉）和 `main`（正本）。前面的 `*` 表示你现在在 `main` 上
- 抽屉里那次存档：`89 files changed, 19132 insertions(+)`：89 个文件、约 1.9 万行，一行没丢

### 面试怎么说
> "Unmerged experimental work went onto an archive branch instead of being deleted or left uncommitted."

---

## 第 3 步 · 旧项目和新项目分开放

### 在干什么
`hearing_agent/` 变成一个普通文件夹，里面放两个独立的项目：

```
hearing_agent/
├── old/                  旧项目（OtoVCE，原样搬进来，带着自己的记录）
└── rare-disease-agent/   新项目（从零开始的记录）
```

### 背景
新项目要从零开始一份干净的记录。如果在旧项目里接着改，新项目的历史里会混进论文的 9 次提交、22 个基因组文件，以后也说不清哪些是新做的。

### 用了什么技术，为什么
**一个仓库 = 一个带 `.git` 的文件夹。** 记录本（隐藏文件夹 `.git`）跟着文件夹走，所以整个搬进 `old/` 之后，旧项目的标签、分支、和 GitHub 的连接都照常工作。两个项目各有各的 `.git`，互不影响。

### 公司里什么情况怎么做
- **大改或重写**：通常新开一个仓库，旧的打标签冻结、只修严重问题，而不是在旧代码上原地大改
- **新项目要用旧代码**：一块一块搬，搬一块配一块测试，不整个复制过去

### 执行

In [ ]:
run("ls ..")
run("git status -sb | head -1")

### 结果怎么看
- `ls ..` 列出上一层文件夹 `hearing_agent/`：正好两个，`old` 和 `rare-disease-agent`
- 旧项目的状态 `## main...origin/main [ahead 5]`：本地正本比 GitHub 多 5 次提交，就是上面第 1 层和第 2 层的差别。搬家之后一切照常

### 面试怎么说
> "The rewrite lives in a new repository with a clean history; the old one is frozen and only referenced."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 钉名字 | git 标签 | 发布和复现都靠它 |
| 2 存抽屉 | git 分支 | 改动不直接碰正本，没合并的工作也不丢 |
| 3 分开放 | 两个独立仓库 | 新项目从干净的记录开始 |

**一句话**：论文那一版钉住了，两个月的工作收好了，新项目可以放心从零开始。

## 面试题

1. **How do you keep a published result reproducible while the code keeps changing?**
   > "Tag the exact commit the result came from, pin data versions and dependencies, and never move the tag. Anyone can check out that tag and rerun."

2. **What's a branch for?**
   > "Isolating a change from the main line. Every change happens on a branch and comes back through a reviewed pull request, so main stays releasable."

**下一章 02 · 新项目骨架**：在新文件夹里搭一个标准的 Python 项目：一条命令装好，一条命令跑起来。